# NEPSE AI Trader — Google Colab

**Run cells 1 → 12 in order.**  
All data saves permanently to your Google Drive.

> Your working NepseAlpha cloudscraper code is in Cell 5, extended with full persistent storage.

## Cell 1 — Install Packages

In [1]:
!pip install -q cloudscraper requests pandas pandas-ta xgboost lightgbm scikit-learn joblib schedule beautifulsoup4 lxml

import subprocess
result = subprocess.run(['pip', 'show', 'cloudscraper'], capture_output=True, text=True)
print("cloudscraper:", "OK" if 'Version' in result.stdout else "FAILED")
print("All packages installed!")

cloudscraper: OK
All packages installed!


## Cell 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All data goes here in your Google Drive
BASE_DIR = '/content/drive/MyDrive/NEPSE_AI'

# Create all needed folders
folders = [
    BASE_DIR,
    f'{BASE_DIR}/data',          # SQLite database
    f'{BASE_DIR}/signals',       # Daily signal CSV exports
    f'{BASE_DIR}/models',        # Trained AI models
    f'{BASE_DIR}/logs',          # Log files
    f'{BASE_DIR}/exports',       # Excel/CSV exports for review
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("Google Drive mounted successfully!")
print(f"Your data folder: {BASE_DIR}")
print("Subfolders created:")
for f in folders:
    print(f"  {f}")

ModuleNotFoundError: No module named 'google.colab'

## Cell 3 — Settings & Config

In [ ]:
import os, sys, logging, io, time, json
from datetime import datetime, date, timedelta

# ── Paths ────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/NEPSE_AI'
DB_PATH     = f'{BASE_DIR}/data/nepse_ai.db'
MODELS_DIR  = f'{BASE_DIR}/models'
SIGNALS_DIR = f'{BASE_DIR}/signals'
EXPORTS_DIR = f'{BASE_DIR}/exports'
LOG_FILE    = f'{BASE_DIR}/logs/nepse.log'

# ── Data sources ─────────────────────────────────────────────
NEPSEALPHA_LIVE_URL   = "https://nepsealpha.com/live/stocks"
NEPALSTOCK_API        = "https://www.nepalstock.com.np/api/nots"
NEPSE_UNOFFICIAL_API  = "https://nepseapi.surajrimal.dev"

# ── Scraping ─────────────────────────────────────────────────
DELAY_MIN = 2.0
DELAY_MAX = 5.0
TIMEOUT   = 20
RETRIES   = 3

# ── Market calendar ───────────────────────────────────────────
# Python weekday: Mon=0 Tue=1 Wed=2 Thu=3 Fri=4 Sat=5 Sun=6
# NEPSE: Sun-Thu → 6,0,1,2,3
MARKET_DAYS       = [6, 0, 1, 2, 3]
MARKET_OPEN_HOUR  = 11
MARKET_CLOSE_HOUR = 15
NEPSE_HOLIDAYS    = [
    "2025-04-14","2025-05-01","2025-08-21","2025-09-19",
    "2025-10-02","2025-10-20","2025-11-05",
    "2026-01-11","2026-02-19","2026-04-14","2026-04-15",
]

# ── Logging ────────────────────────────────────────────────
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', line_buffering=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)-8s] %(name)s: %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
    ]
)
logger = logging.getLogger("NEPSE")

# ── Helpers ───────────────────────────────────────────────────
def is_market_open():
    now = datetime.now()
    if now.weekday() not in MARKET_DAYS: return False
    if now.strftime("%Y-%m-%d") in NEPSE_HOLIDAYS: return False
    return MARKET_OPEN_HOUR <= now.hour < MARKET_CLOSE_HOUR

def is_trading_day(d=None):
    d = d or date.today()
    if d.weekday() not in MARKET_DAYS: return False
    return d.strftime("%Y-%m-%d") not in NEPSE_HOLIDAYS

def today_str(): return date.today().strftime("%Y-%m-%d")
def now_str():   return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Settings loaded OK")
print(f"  DB Path:     {DB_PATH}")
print(f"  Models:      {MODELS_DIR}")
print(f"  Market open: {is_market_open()}")
print(f"  Trading day: {is_trading_day()}")

## Cell 4 — Database (15 tables)

In [ ]:
import sqlite3

TABLES = [
    # Master stock list
    ("stocks", \"\"\"CREATE TABLE IF NOT EXISTS stocks (
        id INTEGER PRIMARY KEY AUTOINCREMENT, symbol TEXT UNIQUE NOT NULL,
        company_name TEXT, sector TEXT, listed_shares REAL,
        paid_up_value REAL, is_active INTEGER DEFAULT 1,
        created_at TEXT, updated_at TEXT)\"\"\"),

    # Live snapshots (every 5 min during market hours)
    ("live_market", \"\"\"CREATE TABLE IF NOT EXISTS live_market (
        id INTEGER PRIMARY KEY AUTOINCREMENT, symbol TEXT NOT NULL,
        ltp REAL, open_price REAL, high_price REAL, low_price REAL,
        close_price REAL, volume INTEGER, turnover REAL,
        pct_change REAL, prev_close REAL,
        snapshot_time TEXT NOT NULL, created_at TEXT)\"\"\"),

    # Daily OHLCV (end of day + historical)
    ("daily_ohlcv", \"\"\"CREATE TABLE IF NOT EXISTS daily_ohlcv (
        id INTEGER PRIMARY KEY AUTOINCREMENT, symbol TEXT NOT NULL,
        trade_date TEXT NOT NULL, open_price REAL, high_price REAL,
        low_price REAL, close_price REAL, adj_close REAL,
        volume INTEGER, turnover REAL, total_trades INTEGER,
        created_at TEXT, UNIQUE(symbol, trade_date))\"\"\"),

    # Full floorsheet (every trade)
    ("floorsheet", \"\"\"CREATE TABLE IF NOT EXISTS floorsheet (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        transaction_id TEXT UNIQUE, trade_date TEXT NOT NULL,
        symbol TEXT NOT NULL, buyer_broker INTEGER,
        seller_broker INTEGER, quantity INTEGER,
        rate REAL, amount REAL, created_at TEXT)\"\"\"),

    # Broker buy/sell per stock per day
    ("broker_activity", \"\"\"CREATE TABLE IF NOT EXISTS broker_activity (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        trade_date TEXT NOT NULL, symbol TEXT NOT NULL,
        broker_number INTEGER NOT NULL,
        buy_qty INTEGER DEFAULT 0, buy_amount REAL DEFAULT 0,
        sell_qty INTEGER DEFAULT 0, sell_amount REAL DEFAULT 0,
        net_qty INTEGER, created_at TEXT,
        UNIQUE(trade_date, symbol, broker_number))\"\"\"),

    # Broker concentration score per stock
    ("broker_concentration", \"\"\"CREATE TABLE IF NOT EXISTS broker_concentration (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        trade_date TEXT NOT NULL, symbol TEXT NOT NULL,
        top3_buy_pct REAL, top3_sell_pct REAL,
        unique_buyers INTEGER, unique_sellers INTEGER,
        accumulation_score REAL, created_at TEXT,
        UNIQUE(trade_date, symbol))\"\"\"),

    # Market indices
    ("indices", \"\"\"CREATE TABLE IF NOT EXISTS indices (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        index_name TEXT NOT NULL, index_date TEXT NOT NULL,
        close_value REAL, pct_change REAL, turnover REAL,
        snapshot_time TEXT, created_at TEXT,
        UNIQUE(index_name, index_date, snapshot_time))\"\"\"),

    # Top movers
    ("top_movers", \"\"\"CREATE TABLE IF NOT EXISTS top_movers (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        trade_date TEXT NOT NULL, snapshot_time TEXT,
        category TEXT NOT NULL, rank_pos INTEGER,
        symbol TEXT NOT NULL, value REAL, ltp REAL,
        pct_change REAL, volume INTEGER, created_at TEXT)\"\"\"),

    # Corporate actions (dividends, bonus, AGM)
    ("corporate_actions", \"\"\"CREATE TABLE IF NOT EXISTS corporate_actions (
        id INTEGER PRIMARY KEY AUTOINCREMENT, symbol TEXT NOT NULL,
        action_type TEXT NOT NULL, announcement_date TEXT,
        ex_date TEXT, book_close_date TEXT,
        value REAL, cash_per_share REAL, remarks TEXT,
        created_at TEXT,
        UNIQUE(symbol, action_type, announcement_date))\"\"\"),

    # Technical indicators
    ("technical_indicators", \"\"\"CREATE TABLE IF NOT EXISTS technical_indicators (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT NOT NULL, calc_date TEXT NOT NULL,
        ema9 REAL, ema21 REAL, ema50 REAL, ema200 REAL,
        macd_line REAL, macd_signal REAL, macd_histogram REAL,
        rsi14 REAL, adx REAL, stoch_k REAL, stoch_d REAL,
        atr14 REAL, bb_upper REAL, bb_middle REAL, bb_lower REAL,
        bb_pct_b REAL, bb_squeeze INTEGER,
        obv REAL, vwap REAL, volume_ratio REAL, cmf REAL,
        signal_strength REAL, created_at TEXT,
        UNIQUE(symbol, calc_date))\"\"\"),

    # News & sentiment
    ("news", \"\"\"CREATE TABLE IF NOT EXISTS news (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        headline TEXT NOT NULL, url TEXT UNIQUE, source TEXT,
        published_at TEXT, symbols_mentioned TEXT,
        category TEXT, sentiment_score REAL, sentiment_label TEXT,
        created_at TEXT)\"\"\"),

    # AI signals (final output)
    ("ai_signals", \"\"\"CREATE TABLE IF NOT EXISTS ai_signals (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        signal_date TEXT NOT NULL, target_date TEXT NOT NULL,
        symbol TEXT NOT NULL, signal TEXT NOT NULL,
        confidence_pct REAL, ltp REAL,
        stop_loss REAL, target_price REAL, risk_reward REAL,
        key_drivers TEXT, model_version TEXT,
        actual_close REAL, actual_pct REAL, correct INTEGER,
        created_at TEXT, UNIQUE(signal_date, symbol))\"\"\"),

    # Scraper run log
    ("scraper_log", \"\"\"CREATE TABLE IF NOT EXISTS scraper_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        module TEXT NOT NULL, status TEXT NOT NULL,
        records INTEGER DEFAULT 0, error TEXT,
        started_at TEXT, finished_at TEXT, created_at TEXT)\"\"\"),
]

INDEXES = [
    "CREATE INDEX IF NOT EXISTS idx_live    ON live_market(symbol, snapshot_time)",
    "CREATE INDEX IF NOT EXISTS idx_ohlcv   ON daily_ohlcv(symbol, trade_date DESC)",
    "CREATE INDEX IF NOT EXISTS idx_floor   ON floorsheet(symbol, trade_date DESC)",
    "CREATE INDEX IF NOT EXISTS idx_broker  ON broker_activity(trade_date DESC, symbol)",
    "CREATE INDEX IF NOT EXISTS idx_conc    ON broker_concentration(trade_date DESC, symbol)",
    "CREATE INDEX IF NOT EXISTS idx_ti      ON technical_indicators(symbol, calc_date DESC)",
    "CREATE INDEX IF NOT EXISTS idx_signals ON ai_signals(signal_date DESC, symbol)",
]


class DB:
    \"\"\"Thin SQLite wrapper.\"\"\"

    def conn(self):
        c = sqlite3.connect(DB_PATH, timeout=30)
        c.row_factory = sqlite3.Row
        c.execute("PRAGMA journal_mode=WAL")
        return c

    def init(self):
        c = self.conn()
        try:
            for _, sql in TABLES:
                c.execute(sql)
            for idx in INDEXES:
                c.execute(idx)
            c.commit()
            print(f"Database ready — {len(TABLES)} tables in {DB_PATH}")
        finally:
            c.close()

    def run(self, sql, params=None):
        c = self.conn()
        try:
            cur = c.cursor()
            cur.execute(sql, params or [])
            c.commit()
            return cur.rowcount
        except Exception as e:
            c.rollback()
            logger.error(f"DB error: {e} | {sql[:100]}")
            raise
        finally:
            c.close()

    def runmany(self, sql, data):
        if not data: return 0
        c = self.conn()
        try:
            cur = c.cursor()
            cur.executemany(sql, data)
            c.commit()
            return cur.rowcount
        except Exception as e:
            c.rollback()
            logger.error(f"DB executemany error: {e}")
            raise
        finally:
            c.close()

    def one(self, sql, params=None):
        c = self.conn()
        try:
            cur = c.cursor()
            cur.execute(sql, params or [])
            r = cur.fetchone()
            return dict(r) if r else None
        finally:
            c.close()

    def all(self, sql, params=None):
        c = self.conn()
        try:
            cur = c.cursor()
            cur.execute(sql, params or [])
            return [dict(r) for r in cur.fetchall()]
        finally:
            c.close()

    def log(self, module, status, records=0, error=None, started=None):
        try:
            self.run(
                \"\"\"INSERT INTO scraper_log
                  (module, status, records, error, started_at, finished_at, created_at)
                  VALUES (?,?,?,?,?,?,?)\"\"\",
                [module, status, records, error, started, now_str(), now_str()]
            )
        except Exception:
            pass


db = DB()
db.init()
print("DB object created: db")

## Cell 5 — NepseAlpha Scraper (your cloudscraper, extended)

In [ ]:
import cloudscraper, random, time as _time
from collections import defaultdict

class NEPSEAlphaScraper:
    \"\"\"
    Extended version of your working cloudscraper approach.
    Hits NepseAlpha's internal endpoints directly.
    \"\"\"

    LIVE_URL       = "https://nepsealpha.com/live/stocks"
    TRADING_URL    = "https://nepsealpha.com/trading/1/data"
    TECH_SIG_URL   = "https://nepsealpha.com/trading-signals/tech"

    UA_LIST = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    ]

    def __init__(self):
        self.scraper = cloudscraper.create_scraper(
            browser={"browser": "chrome", "platform": "windows", "mobile": False}
        )
        self.scraper.headers.update({
            "Accept":          "application/json, text/plain, */*",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer":         "https://nepsealpha.com/live-market",
        })

    def _get(self, url, params=None, retries=RETRIES):
        \"\"\"GET with retry + polite delay.\"\"\"
        for attempt in range(1, retries + 1):
            try:
                self.scraper.headers["User-Agent"] = random.choice(self.UA_LIST)
                resp = self.scraper.get(url, params=params, timeout=TIMEOUT)
                if resp.status_code == 200:
                    return resp
                logger.warning(f"HTTP {resp.status_code} on attempt {attempt}: {url}")
            except Exception as e:
                logger.warning(f"Request error attempt {attempt}: {e}")
            _time.sleep(DELAY_MIN * attempt)
        return None

    # ── MODULE 1: Live Market (YOUR WORKING CODE, EXTENDED) ────
    def fetch_live_market(self):
        \"\"\"
        Fetch all live stock prices from NepseAlpha.
        Uses your exact working endpoint and params.
        \"\"\"
        logger.info("Fetching live market from NepseAlpha...")
        start = now_str()

        params = {
            "fsk":             str(int(_time.time() * 1000)),
            "selectedSector":  "NEPSE",
            "sector":          "",
            "fs":              "123100"
        }

        resp = self._get(self.LIVE_URL, params=params)
        if not resp:
            db.log("live_market", "failed", 0, "No response", start)
            return []

        try:
            data   = resp.json()
            prices = data.get("stock_live", {}).get("prices", [])
        except Exception as e:
            logger.error(f"JSON parse error: {e}")
            db.log("live_market", "failed", 0, str(e), start)
            return []

        if not prices:
            logger.warning("Empty prices list from NepseAlpha")
            return []

        # ── Save to DB ────────────────────────────────────────
        snap = datetime.now().strftime("%Y-%m-%d %H:%M")
        now  = now_str()
        records = []

        for item in prices:
            sym = (item.get("symbol") or "").strip().upper()
            if not sym:
                continue
            records.append((
                sym,
                item.get("close")   or item.get("ltp")  or 0,
                item.get("open")    or 0,
                item.get("high")    or 0,
                item.get("low")     or 0,
                item.get("close")   or 0,
                item.get("volume")  or item.get("qty") or 0,
                item.get("turn_over") or item.get("turnover") or 0,
                item.get("percent_change") or item.get("change") or 0,
                item.get("prev_close") or 0,
                snap, now,
            ))

        if records:
            db.runmany(\"\"\"
                INSERT INTO live_market
                  (symbol, ltp, open_price, high_price, low_price, close_price,
                   volume, turnover, pct_change, prev_close, snapshot_time, created_at)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
            \"\"\", records)

            # Also upsert stocks table
            stock_recs = [(r[0], r[0], "Unknown", 0, 100, 1, now, now)
                          for r in records]
            db.runmany(\"\"\"
                INSERT INTO stocks
                  (symbol, company_name, sector, listed_shares,
                   paid_up_value, is_active, created_at, updated_at)
                VALUES (?,?,?,?,?,?,?,?)
                ON CONFLICT(symbol) DO UPDATE SET updated_at=excluded.updated_at
            \"\"\", stock_recs)

        logger.info(f"Live market: {len(records)} stocks saved at {snap}")
        db.log("live_market", "success", len(records), None, start)
        return [{"symbol":r[0],"ltp":r[1],"open":r[2],"high":r[3],"low":r[4],
                 "close":r[5],"volume":r[6],"turnover":r[7],
                 "pct_change":r[8],"prev_close":r[9]} for r in records]

    # ── MODULE 2: EOD OHLCV save (after market close) ──────────
    def save_eod_ohlcv(self):
        \"\"\"
        After market close, take the last live snapshot and save
        as the official end-of-day OHLCV record.
        \"\"\"
        logger.info("Saving EOD OHLCV from last live snapshot...")
        start = now_str()
        today = today_str()

        # Get latest snapshot from live_market
        rows = db.all(\"\"\"
            SELECT symbol, open_price, high_price, low_price,
                   close_price, volume, turnover
            FROM live_market
            WHERE snapshot_time=(SELECT MAX(snapshot_time) FROM live_market
                                 WHERE snapshot_time LIKE ?)
            GROUP BY symbol
        \"\"\", [f"{today}%"])

        if not rows:
            logger.warning("No live data to convert to EOD")
            return 0

        now = now_str()
        records = [(
            r["symbol"], today,
            r["open_price"], r["high_price"], r["low_price"],
            r["close_price"], r["close_price"],    # adj_close = close for now
            r["volume"], r["turnover"], 0, now,
        ) for r in rows]

        saved = db.runmany(\"\"\"
            INSERT INTO daily_ohlcv
              (symbol, trade_date, open_price, high_price, low_price,
               close_price, adj_close, volume, turnover, total_trades, created_at)
            VALUES (?,?,?,?,?,?,?,?,?,?,?)
            ON CONFLICT(symbol, trade_date) DO UPDATE SET
                high_price=MAX(daily_ohlcv.high_price, excluded.high_price),
                low_price=MIN(daily_ohlcv.low_price, excluded.low_price),
                close_price=excluded.close_price,
                volume=excluded.volume
        \"\"\", records)

        logger.info(f"EOD OHLCV saved: {len(records)} stocks for {today}")
        db.log("eod_ohlcv", "success", len(records), None, start)
        return len(records)

    # ── Historical OHLCV via nepalstock official ───────────────
    def fetch_history(self, symbol: str, years: int = 3) -> int:
        \"\"\"Pull historical candles from nepalstock.com.np official API.\"\"\"
        end   = date.today()
        start = end - timedelta(days=years * 365)

        url    = f"{NEPALSTOCK_API}/graphData/historical"
        params = {
            "symbol":    symbol,
            "startDate": start.strftime("%Y-%m-%d"),
            "endDate":   end.strftime("%Y-%m-%d"),
        }

        resp = self._get(url, params=params)
        if not resp:
            return 0

        try:
            raw = resp.json()
        except Exception:
            return 0

        candles = raw if isinstance(raw, list) else raw.get("data", [])
        now     = now_str()
        records = []

        for c in candles:
            try:
                td    = (c.get("businessDate") or c.get("date") or "")[:10]
                close = float(c.get("closingPrice") or c.get("close") or 0)
                if not td or not close:
                    continue
                records.append((
                    symbol, td,
                    float(c.get("openPrice") or c.get("open") or 0),
                    float(c.get("highPrice") or c.get("high") or 0),
                    float(c.get("lowPrice")  or c.get("low")  or 0),
                    close, close,
                    int(c.get("totalTradedQuantity") or c.get("volume") or 0),
                    float(c.get("totalTurnover") or c.get("turnover") or 0),
                    int(c.get("totalTransactions") or 0),
                    now,
                ))
            except Exception:
                continue

        if records:
            db.runmany(\"\"\"
                INSERT INTO daily_ohlcv
                  (symbol, trade_date, open_price, high_price, low_price,
                   close_price, adj_close, volume, turnover, total_trades, created_at)
                VALUES (?,?,?,?,?,?,?,?,?,?,?)
                ON CONFLICT(symbol, trade_date) DO NOTHING
            \"\"\", records)

        return len(records)

    def fetch_all_history(self):
        \"\"\"Pull 3-year history for all stocks in DB.\"\"\"
        stocks = db.all("SELECT symbol FROM stocks WHERE is_active=1")
        total  = 0
        for i, row in enumerate(stocks, 1):
            sym = row["symbol"]
            n   = self.fetch_history(sym)
            total += n
            logger.info(f"[{i}/{len(stocks)}] {sym}: {n} candles")
            _time.sleep(random.uniform(1.5, 3.0))
        logger.info(f"History complete: {total} total candles")
        return total

    # ── MODULE 5: Indices ──────────────────────────────────────
    def fetch_indices(self):
        \"\"\"Fetch NEPSE + sector indices from unofficial API.\"\"\"
        logger.info("Fetching indices...")
        start   = now_str()
        today   = today_str()
        snap    = datetime.now().strftime("%H:%M")
        records = []
        now     = now_str()

        for endpoint in ["/NepseIndex", "/SubIndices"]:
            resp = self._get(f"{NEPSE_UNOFFICIAL_API}{endpoint}")
            if not resp:
                continue
            try:
                data  = resp.json()
                items = data if isinstance(data, list) else data.get("data", [data])
                for idx in items:
                    name = idx.get("indexName") or idx.get("index") or "NEPSE"
                    records.append((
                        name, today,
                        idx.get("currentValue") or idx.get("closeValue") or 0,
                        idx.get("percentageChange") or 0,
                        idx.get("turnover") or 0,
                        snap, now,
                    ))
            except Exception as e:
                logger.warning(f"Index parse error: {e}")
            _time.sleep(1)

        if records:
            db.runmany(\"\"\"
                INSERT INTO indices
                  (index_name, index_date, close_value, pct_change,
                   turnover, snapshot_time, created_at)
                VALUES (?,?,?,?,?,?,?)
                ON CONFLICT(index_name, index_date, snapshot_time) DO NOTHING
            \"\"\", records)

        logger.info(f"Indices: {len(records)} snapshots saved")
        db.log("indices", "success", len(records), None, start)
        return len(records)

    # ── MODULE 6: Top Movers ───────────────────────────────────
    def fetch_top_movers(self):
        logger.info("Fetching top movers...")
        start   = now_str()
        today   = today_str()
        snap    = datetime.now().strftime("%H:%M")
        records = []
        now     = now_str()

        endpoints = {
            "gainer":  "/TopGainers",
            "loser":   "/TopLosers",
            "volume":  "/TopStocksByVolume",
            "turnover":"/TopStocksByTurnover",
        }

        for cat, ep in endpoints.items():
            resp = self._get(f"{NEPSE_UNOFFICIAL_API}{ep}")
            if not resp:
                continue
            try:
                data  = resp.json()
                items = data if isinstance(data, list) else data.get("data", [])
                for rank, item in enumerate(items[:20], 1):
                    sym = (item.get("symbol") or "").strip().upper()
                    if not sym: continue
                    records.append((
                        today, snap, cat, rank, sym,
                        float(item.get("percentageChange") or item.get("volume") or 0),
                        float(item.get("lastTradedPrice")  or item.get("ltp") or 0),
                        float(item.get("percentageChange") or 0),
                        int(item.get("totalTradeQuantity") or 0),
                        now,
                    ))
            except Exception as e:
                logger.warning(f"Top mover parse error: {e}")
            _time.sleep(0.5)

        if records:
            db.runmany(\"\"\"
                INSERT INTO top_movers
                  (trade_date, snapshot_time, category, rank_pos,
                   symbol, value, ltp, pct_change, volume, created_at)
                VALUES (?,?,?,?,?,?,?,?,?,?)
            \"\"\", records)

        logger.info(f"Top movers: {len(records)} entries")
        db.log("top_movers", "success", len(records), None, start)
        return len(records)

    # ── MODULE 7: Corporate Actions ────────────────────────────
    def fetch_corporate_actions(self):
        logger.info("Fetching corporate actions...")
        start = now_str()
        total = 0
        now   = now_str()

        for action_type, endpoint in [
            ("dividend",  "/dividendList"),
            ("bonus",     "/bonusList"),
            ("agm",       "/agmList"),
            ("bookclose", "/bookCloseList"),
        ]:
            resp = self._get(f"{NEPALSTOCK_API}{endpoint}")
            if not resp:
                continue
            try:
                raw   = resp.json()
                items = raw if isinstance(raw, list) else raw.get("data", [])
                recs  = []
                for item in items:
                    sym = (item.get("symbol") or "").strip().upper()
                    if not sym: continue
                    recs.append((
                        sym, action_type,
                        item.get("announcementDate") or "",
                        item.get("exDate") or item.get("fromDate") or "",
                        item.get("bookCloseDate") or item.get("toDate") or "",
                        float(item.get("dividendPercentage") or
                              item.get("bonusPercentage") or 0),
                        float(item.get("cashDividendPerUnit") or 0),
                        item.get("remarks") or "",
                        now,
                    ))
                if recs:
                    db.runmany(\"\"\"
                        INSERT INTO corporate_actions
                          (symbol, action_type, announcement_date, ex_date,
                           book_close_date, value, cash_per_share, remarks, created_at)
                        VALUES (?,?,?,?,?,?,?,?,?)
                        ON CONFLICT(symbol, action_type, announcement_date) DO NOTHING
                    \"\"\", recs)
                    total += len(recs)
                    logger.info(f"  {action_type}: {len(recs)}")
            except Exception as e:
                logger.warning(f"Corp action error ({action_type}): {e}")
            _time.sleep(1)

        db.log("corporate_actions", "success", total, None, start)
        return total


# Create instance
scraper = NEPSEAlphaScraper()
print("NEPSEAlphaScraper ready")
print(f"  Live URL: {scraper.LIVE_URL}")

## Cell 6 — Floorsheet + Broker Scraper

In [ ]:
from collections import defaultdict

class FloorsheetScraper:
    \"\"\"Module 3 + 4: Floorsheet collection and broker analysis.\"\"\"

    def fetch_floorsheet(self, trade_date=None):
        trade_date = trade_date or today_str()
        logger.info(f"Fetching floorsheet for {trade_date}...")
        start = now_str()
        all_rows = []
        page = 1

        while True:
            resp = scraper._get(
                f"{NEPALSTOCK_API}/floorsheet",
                params={"page": page, "size": 2000,
                        "businessDate": trade_date, "sort": "contractId,desc"}
            )
            if not resp:
                break

            try:
                data    = resp.json()
                content = (data.get("floorsheets", {}).get("content")
                           or data.get("data")
                           or (data if isinstance(data, list) else []))
                if not content:
                    break

                now = now_str()
                for row in content:
                    tid = str(row.get("contractId") or row.get("transactionId") or "")
                    sym = (row.get("stockSymbol") or row.get("symbol") or "").strip().upper()
                    if not sym: continue
                    tid = tid or f"{trade_date}_{sym}_{len(all_rows)}"
                    all_rows.append((
                        tid, trade_date, sym,
                        int(row.get("buyerMemberId")  or row.get("buyerBroker")  or 0),
                        int(row.get("sellerMemberId") or row.get("sellerBroker") or 0),
                        int(row.get("contractQuantity") or row.get("quantity") or 0),
                        float(row.get("contractRate")   or row.get("rate")   or 0),
                        float(row.get("contractAmount") or row.get("amount") or 0),
                        now,
                    ))

                total_pages = (data.get("floorsheets", {}).get("totalPages")
                               or data.get("totalPages") or 1)
                logger.info(f"  Page {page}/{total_pages} — {len(content)} trades")
                if page >= total_pages:
                    break
                page += 1
                import time as _t; _t.sleep(1)

            except Exception as e:
                logger.error(f"Floorsheet page {page} error: {e}")
                break

        if all_rows:
            db.runmany(\"\"\"
                INSERT INTO floorsheet
                  (transaction_id, trade_date, symbol, buyer_broker,
                   seller_broker, quantity, rate, amount, created_at)
                VALUES (?,?,?,?,?,?,?,?,?)
                ON CONFLICT(transaction_id) DO NOTHING
            \"\"\", all_rows)

        logger.info(f"Floorsheet: {len(all_rows)} trades for {trade_date}")
        db.log("floorsheet", "success", len(all_rows), None, start)
        return len(all_rows)

    def compute_broker_stats(self, trade_date=None):
        trade_date = trade_date or today_str()
        logger.info(f"Computing broker stats for {trade_date}...")
        start = now_str()

        rows = db.all(\"\"\"
            SELECT symbol, buyer_broker, seller_broker, quantity, amount
            FROM floorsheet WHERE trade_date=?
        \"\"\", [trade_date])

        if not rows:
            logger.warning(f"No floorsheet data for {trade_date}")
            return 0

        buy  = defaultdict(lambda: defaultdict(lambda: [0, 0.0]))
        sell = defaultdict(lambda: defaultdict(lambda: [0, 0.0]))

        for r in rows:
            sym = r["symbol"]
            qty = r["quantity"] or 0
            amt = r["amount"] or 0
            if r["buyer_broker"]:
                buy[sym][r["buyer_broker"]][0] += qty
                buy[sym][r["buyer_broker"]][1] += amt
            if r["seller_broker"]:
                sell[sym][r["seller_broker"]][0] += qty
                sell[sym][r["seller_broker"]][1] += amt

        all_syms = set(list(buy) + list(sell))
        act_recs  = []
        conc_recs = []
        now = now_str()

        for sym in all_syms:
            all_brokers = set(list(buy[sym]) + list(sell[sym]))
            for br in all_brokers:
                bq, ba = buy[sym].get(br, [0, 0.0])
                sq, sa = sell[sym].get(br, [0, 0.0])
                act_recs.append((trade_date, sym, br,
                                 bq, ba, sq, sa, bq-sq, now))

            # Concentration
            buys_sorted  = sorted(buy[sym].items(),  key=lambda x: x[1][0], reverse=True)
            sells_sorted = sorted(sell[sym].items(), key=lambda x: x[1][0], reverse=True)
            total_buy    = sum(v[0] for v in buy[sym].values())
            total_sell   = sum(v[0] for v in sell[sym].values())
            top3_buy     = sum(v[0] for _, v in buys_sorted[:3])
            top3_sell    = sum(v[0] for _, v in sells_sorted[:3])
            top3_buy_pct  = (top3_buy  / total_buy  * 100) if total_buy  else 0
            top3_sell_pct = (top3_sell / total_sell * 100) if total_sell else 0
            score = round(top3_buy_pct - top3_sell_pct, 2)

            conc_recs.append((
                trade_date, sym,
                round(top3_buy_pct, 2), round(top3_sell_pct, 2),
                len(buy[sym]), len(sell[sym]),
                score, now,
            ))

        if act_recs:
            db.runmany(\"\"\"
                INSERT INTO broker_activity
                  (trade_date, symbol, broker_number,
                   buy_qty, buy_amount, sell_qty, sell_amount, net_qty, created_at)
                VALUES (?,?,?,?,?,?,?,?,?)
                ON CONFLICT(trade_date, symbol, broker_number) DO UPDATE SET
                    buy_qty=excluded.buy_qty, sell_qty=excluded.sell_qty
            \"\"\", act_recs)

        if conc_recs:
            db.runmany(\"\"\"
                INSERT INTO broker_concentration
                  (trade_date, symbol, top3_buy_pct, top3_sell_pct,
                   unique_buyers, unique_sellers, accumulation_score, created_at)
                VALUES (?,?,?,?,?,?,?,?)
                ON CONFLICT(trade_date, symbol) DO UPDATE SET
                    accumulation_score=excluded.accumulation_score
            \"\"\", conc_recs)

        logger.info(f"Broker stats: {len(all_syms)} stocks analyzed")
        db.log("broker_stats", "success", len(act_recs), None, start)
        return len(act_recs)


floor_scraper = FloorsheetScraper()
print("FloorsheetScraper ready")

## Cell 7 — Technical Indicator Engine

In [ ]:
import pandas as pd
import numpy as np

try:
    import pandas_ta as ta
    TA_AVAILABLE = True
    print("pandas_ta available")
except ImportError:
    TA_AVAILABLE = False
    print("WARNING: pandas_ta not installed — run Cell 1 first")


class IndicatorEngine:
    \"\"\"Module 8: Compute all technical indicators from OHLCV data.\"\"\"

    def compute_all(self, calc_date=None):
        if not TA_AVAILABLE:
            print("pandas_ta required. Run: !pip install pandas-ta")
            return 0

        calc_date = calc_date or today_str()
        logger.info(f"Computing indicators for {calc_date}...")
        start  = now_str()
        stocks = db.all("SELECT symbol FROM stocks WHERE is_active=1")
        ok, err = 0, 0

        for row in stocks:
            try:
                if self._compute(row["symbol"], calc_date):
                    ok += 1
            except Exception as e:
                err += 1
                logger.debug(f"{row['symbol']} indicator error: {e}")

        logger.info(f"Indicators: {ok} computed, {err} errors")
        db.log("indicators", "success", ok, None, start)
        return ok

    def _compute(self, symbol, calc_date):
        rows = db.all(\"\"\"
            SELECT trade_date, open_price, high_price, low_price,
                   close_price, volume
            FROM daily_ohlcv
            WHERE symbol=? AND trade_date<=?
            ORDER BY trade_date DESC LIMIT 250
        \"\"\", [symbol, calc_date])

        if len(rows) < 25:
            return False

        df = pd.DataFrame(rows[::-1])
        df.columns = ["date","open","high","low","close","volume"]
        df = df.astype({"open":float,"high":float,"low":float,
                        "close":float,"volume":float})
        df.set_index("date", inplace=True)

        # Trend
        df["ema9"]  = ta.ema(df["close"], length=9)
        df["ema21"] = ta.ema(df["close"], length=21)
        df["ema50"] = ta.ema(df["close"], length=50)
        df["ema200"]= ta.ema(df["close"], length=200)

        macd = ta.macd(df["close"])
        if macd is not None and not macd.empty:
            df["macd_l"] = macd.iloc[:,0]
            df["macd_s"] = macd.iloc[:,1]
            df["macd_h"] = macd.iloc[:,2]

        adx_df = ta.adx(df["high"], df["low"], df["close"])
        if adx_df is not None and not adx_df.empty:
            df["adx"] = adx_df.iloc[:,0]

        # Momentum
        df["rsi"]  = ta.rsi(df["close"], length=14)
        stoch = ta.stoch(df["high"], df["low"], df["close"])
        if stoch is not None and not stoch.empty:
            df["stoch_k"] = stoch.iloc[:,0]
            df["stoch_d"] = stoch.iloc[:,1]

        # Volatility
        df["atr"] = ta.atr(df["high"], df["low"], df["close"])
        bb = ta.bbands(df["close"])
        bb_u = bb_m = bb_l = bb_p = None
        bb_sq = 0
        if bb is not None and not bb.empty:
            bb_l  = bb.iloc[:,0]; bb_m = bb.iloc[:,1]
            bb_u  = bb.iloc[:,2]; bb_p = bb.iloc[:,4]
            bw    = (bb_u - bb_l) / bb_m.replace(0, np.nan)
            bb_sq = int(bw.iloc[-1] < bw.rolling(20).mean().iloc[-1] * 0.75
                        if len(bw) >= 20 else 0)

        # Volume
        df["obv"]     = ta.obv(df["close"], df["volume"])
        df["vol_ma"]  = ta.sma(df["volume"], length=20)
        df["vol_rat"] = df["volume"] / df["vol_ma"].replace(0, np.nan)
        try:
            df["vwap"] = ta.vwap(df["high"], df["low"], df["close"], df["volume"])
        except Exception:
            df["vwap"] = df["close"]
        df["cmf"] = ta.cmf(df["high"], df["low"], df["close"], df["volume"])

        last   = df.iloc[-1]
        score  = self._score(last, df)

        def s(v):
            try:
                f = float(v); return round(f,4) if f==f else None
            except: return None

        db.run(\"\"\"
            INSERT INTO technical_indicators
              (symbol, calc_date, ema9, ema21, ema50, ema200,
               macd_line, macd_signal, macd_histogram, rsi14, adx,
               stoch_k, stoch_d, atr14,
               bb_upper, bb_middle, bb_lower, bb_pct_b, bb_squeeze,
               obv, vwap, volume_ratio, cmf, signal_strength, created_at)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
            ON CONFLICT(symbol, calc_date) DO UPDATE SET
                rsi14=excluded.rsi14,
                signal_strength=excluded.signal_strength
        \"\"\", [
            symbol, calc_date,
            s(last.get("ema9")),  s(last.get("ema21")),
            s(last.get("ema50")), s(last.get("ema200")),
            s(last.get("macd_l")), s(last.get("macd_s")),
            s(last.get("macd_h")), s(last.get("rsi")),
            s(last.get("adx")),   s(last.get("stoch_k")),
            s(last.get("stoch_d")), s(last.get("atr")),
            s(bb_u.iloc[-1] if bb_u is not None else None),
            s(bb_m.iloc[-1] if bb_m is not None else None),
            s(bb_l.iloc[-1] if bb_l is not None else None),
            s(bb_p.iloc[-1] if bb_p is not None else None),
            bb_sq,
            s(last.get("obv")), s(last.get("vwap")),
            s(last.get("vol_rat")), s(last.get("cmf")),
            round(score, 2),
            now_str(),
        ])
        return True

    def _score(self, last, df):
        score, w = 0.0, 0

        def add(v, weight):
            nonlocal score, w
            try:
                f = float(v)
                if f == f: score += f * weight; w += weight
            except: pass

        rsi = last.get("rsi")
        if rsi: add((50 - float(rsi)) / 50 * 100, 2.0)

        hist = last.get("macd_h")
        if hist: add(min(max(float(hist), -100), 100), 1.5)

        close  = float(df["close"].iloc[-1])
        ema50  = last.get("ema50")
        if ema50 and float(ema50) > 0:
            add(100 if close > float(ema50) else -100, 2.0)

        vol_r = last.get("vol_rat")
        if vol_r and len(df) > 1:
            prev = float(df["close"].iloc[-2])
            direction = 1 if close >= prev else -1
            if float(vol_r) > 1.5: add(50 * direction, 1.0)

        return max(-100, min(100, score / w)) if w > 0 else 0.0


indicator_engine = IndicatorEngine()
print("IndicatorEngine ready")

## Cell 8 — AI Signal Engine

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

class AISignalEngine:
    \"\"\"
    Combines all data modules into AI signals.
    Two modes:
      1. Rule-based (works immediately, no training needed)
      2. ML-based (XGBoost + LightGBM after 30+ days of data)
    \"\"\"
    VERSION = "1.0"

    def __init__(self):
        self.xgb  = None
        self.lgbm = None
        self.feat_cols = []
        self._load_models()

    # ── RULE-BASED SIGNALS (works from day 1) ─────────────────
    def rule_based_signals(self, signal_date=None):
        \"\"\"
        Generate signals using technical + broker rules.
        No training required — works immediately.
        \"\"\"
        signal_date = signal_date or today_str()
        logger.info(f"Generating rule-based signals for {signal_date}...")
        start = now_str()

        # Get latest indicators
        ti_rows = db.all(\"\"\"
            SELECT t.symbol, t.rsi14, t.macd_histogram, t.signal_strength,
                   t.volume_ratio, t.bb_squeeze, t.ema9, t.ema21, t.ema50,
                   t.bb_pct_b, t.adx,
                   l.ltp, l.pct_change, l.volume, l.turnover,
                   b.accumulation_score, b.top3_buy_pct
            FROM technical_indicators t
            LEFT JOIN (
                SELECT symbol, ltp, pct_change, volume, turnover
                FROM live_market
                WHERE snapshot_time=(SELECT MAX(snapshot_time) FROM live_market)
            ) l ON l.symbol=t.symbol
            LEFT JOIN broker_concentration b
                ON b.symbol=t.symbol AND b.trade_date=?
            WHERE t.calc_date=?
        \"\"\", [signal_date, signal_date])

        if not ti_rows:
            # Fallback: use live market data with simple rules
            ti_rows = db.all(\"\"\"
                SELECT symbol, ltp, pct_change, volume, 0 as rsi14,
                       0 as signal_strength, 0 as accumulation_score
                FROM live_market
                WHERE snapshot_time=(SELECT MAX(snapshot_time) FROM live_market)
            \"\"\")

        signals  = []
        now      = now_str()
        target   = self._next_trading_day(signal_date)

        for row in ti_rows:
            sym   = row["symbol"]
            ltp   = float(row.get("ltp") or 0)
            rsi   = float(row.get("rsi14") or 50)
            score = float(row.get("signal_strength") or 0)
            accum = float(row.get("accumulation_score") or 0)
            vol_r = float(row.get("volume_ratio") or 1)
            pct   = float(row.get("pct_change") or 0)
            bb_sq = int(row.get("bb_squeeze") or 0)
            top3  = float(row.get("top3_buy_pct") or 0)
            macd_h= float(row.get("macd_histogram") or 0)

            # ── Rule scoring ──────────────────────────────────
            buy_score  = 0
            sell_score = 0
            drivers    = []

            # RSI rules
            if rsi < 30:
                buy_score  += 3
                drivers.append(f"RSI oversold ({rsi:.0f})")
            elif rsi < 40:
                buy_score  += 1
                drivers.append(f"RSI low ({rsi:.0f})")
            elif rsi > 70:
                sell_score += 3
                drivers.append(f"RSI overbought ({rsi:.0f})")
            elif rsi > 60:
                sell_score += 1

            # MACD
            if macd_h > 0:
                buy_score  += 2
                drivers.append("MACD bullish")
            elif macd_h < 0:
                sell_score += 2
                drivers.append("MACD bearish")

            # Signal strength (composite from indicators)
            if score > 30:
                buy_score  += 2
                drivers.append(f"Strong indicator score (+{score:.0f})")
            elif score > 15:
                buy_score  += 1
            elif score < -30:
                sell_score += 2
                drivers.append(f"Weak indicator score ({score:.0f})")

            # Broker accumulation (NEPSE's biggest edge)
            if accum > 30:
                buy_score  += 3
                drivers.append(f"Broker accumulation: +{accum:.1f}")
            elif accum > 15:
                buy_score  += 2
                drivers.append(f"Mild accumulation: +{accum:.1f}")
            elif accum < -30:
                sell_score += 3
                drivers.append(f"Broker distribution: {accum:.1f}")

            # Volume
            if vol_r > 2.5:
                buy_score += 2 if pct > 0 else 0
                sell_score += 2 if pct < 0 else 0
                drivers.append(f"Volume spike: {vol_r:.1f}x")

            # Bollinger Band squeeze
            if bb_sq:
                buy_score += 1
                drivers.append("BB squeeze (breakout imminent)")

            # Top broker concentration
            if top3 > 65:
                buy_score += 2
                drivers.append(f"Top-3 brokers: {top3:.0f}% buying")

            # ── Classify ──────────────────────────────────────
            total  = buy_score + sell_score
            if total == 0:
                signal = "HOLD"
                conf   = 50.0
            elif buy_score > sell_score * 1.5:
                signal = "BUY"
                conf   = min(95, 50 + buy_score * 5)
            elif sell_score > buy_score * 1.5:
                signal = "SELL"
                conf   = min(95, 50 + sell_score * 5)
            else:
                signal = "HOLD"
                conf   = 50.0

            # ── ATR-based stop loss & target ──────────────────
            atr = ltp * 0.025  # default 2.5% if no ATR
            sl  = round(ltp - 1.5 * atr, 2)
            tp  = round(ltp + 3.0 * atr, 2)
            rr  = round((tp - ltp) / (ltp - sl + 1e-9), 2)

            signals.append({
                "signal_date":   signal_date,
                "target_date":   target,
                "symbol":        sym,
                "signal":        signal,
                "confidence_pct":round(conf, 1),
                "ltp":           ltp,
                "stop_loss":     sl,
                "target_price":  tp,
                "risk_reward":   rr,
                "key_drivers":   drivers[:5],
                "model_version": f"{self.VERSION}-rules",
            })

        # Sort: BUY first by confidence
        signals.sort(key=lambda x: (x["signal"]=="BUY", x["confidence_pct"]), reverse=True)

        # Save to DB
        self._save(signals, now)

        # Export to CSV
        self._export_csv(signals, signal_date)

        buy  = sum(1 for s in signals if s["signal"]=="BUY")
        sell = sum(1 for s in signals if s["signal"]=="SELL")
        logger.info(f"Signals: {buy} BUY | {sell} SELL | {len(signals)-buy-sell} HOLD")
        db.log("signals", "success", len(signals), None, start)
        return signals

    def _save(self, signals, now):
        records = [(
            s["signal_date"], s["target_date"], s["symbol"],
            s["signal"], s["confidence_pct"], s["ltp"],
            s["stop_loss"], s["target_price"], s["risk_reward"],
            json.dumps(s["key_drivers"]), s["model_version"],
            None, None, None, now,
        ) for s in signals]

        if records:
            db.runmany(\"\"\"
                INSERT INTO ai_signals
                  (signal_date, target_date, symbol, signal,
                   confidence_pct, ltp, stop_loss, target_price, risk_reward,
                   key_drivers, model_version,
                   actual_close, actual_pct, correct, created_at)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                ON CONFLICT(signal_date, symbol) DO UPDATE SET
                    signal=excluded.signal,
                    confidence_pct=excluded.confidence_pct
            \"\"\", records)

    def _export_csv(self, signals, signal_date):
        \"\"\"Save signals to CSV in Google Drive exports folder.\"\"\"
        try:
            df   = pd.DataFrame(signals)
            path = f"{EXPORTS_DIR}/signals_{signal_date}.csv"
            df.to_csv(path, index=False)
            logger.info(f"Signals exported: {path}")

            # Also export BUY-only for easy reference
            buys = df[df["signal"]=="BUY"].copy()
            if not buys.empty:
                bpath = f"{EXPORTS_DIR}/BUY_signals_{signal_date}.csv"
                buys.to_csv(bpath, index=False)
                logger.info(f"BUY signals: {bpath}")
        except Exception as e:
            logger.warning(f"CSV export error: {e}")

    def update_backtest(self, start_date=None, end_date=None):
        \"\"\"Compare past signals to actual prices. Updates DB.\"\"\"
        end_date   = end_date   or today_str()
        start_date = start_date or (date.today()-timedelta(days=90)).strftime("%Y-%m-%d")

        pending = db.all(\"\"\"
            SELECT id, symbol, signal, target_date
            FROM ai_signals
            WHERE signal_date BETWEEN ? AND ?
              AND signal IN ('BUY','SELL') AND correct IS NULL
        \"\"\", [start_date, end_date])

        updated = 0
        for sig in pending:
            actual = db.one(
                "SELECT close_price FROM daily_ohlcv WHERE symbol=? AND trade_date=?",
                [sig["symbol"], sig["target_date"]])
            prev = db.one(\"\"\"
                SELECT close_price FROM daily_ohlcv
                WHERE symbol=? AND trade_date<? ORDER BY trade_date DESC LIMIT 1
            \"\"\", [sig["symbol"], sig["target_date"]])

            if not actual or not prev or not prev.get("close_price"): continue

            pct = (actual["close_price"] / prev["close_price"] - 1) * 100
            ok  = 1 if (sig["signal"]=="BUY"  and pct >= 2.0) or \\
                       (sig["signal"]=="SELL" and pct <= -2.0) else 0
            db.run(
                "UPDATE ai_signals SET actual_close=?, actual_pct=?, correct=? WHERE id=?",
                [actual["close_price"], round(pct,3), ok, sig["id"]])
            updated += 1

        logger.info(f"Backtest updated: {updated} signals evaluated")

        stats = db.one(\"\"\"
            SELECT COUNT(*) AS total,
                   ROUND(AVG(correct)*100,1) AS accuracy,
                   ROUND(AVG(CASE WHEN signal='BUY' THEN correct END)*100,1) AS buy_acc,
                   ROUND(AVG(CASE WHEN correct=1 THEN actual_pct END),2) AS avg_win,
                   ROUND(AVG(CASE WHEN correct=0 THEN actual_pct END),2) AS avg_loss
            FROM ai_signals WHERE correct IS NOT NULL
        \"\"\")
        if stats and stats.get("total",0) > 0:
            print(f"\\n=== BACKTEST RESULTS ===")
            print(f"Total signals evaluated: {stats['total']}")
            print(f"Overall accuracy:        {stats['accuracy']}%")
            print(f"BUY accuracy:            {stats['buy_acc']}%")
            print(f"Avg winning trade:       +{stats['avg_win']}%")
            print(f"Avg losing trade:         {stats['avg_loss']}%")
        return stats

    def _next_trading_day(self, from_date):
        d = date.fromisoformat(from_date) + timedelta(days=1)
        for _ in range(14):
            if d.weekday() in MARKET_DAYS and d.strftime("%Y-%m-%d") not in NEPSE_HOLIDAYS:
                return d.strftime("%Y-%m-%d")
            d += timedelta(days=1)
        return d.strftime("%Y-%m-%d")

    def _load_models(self):
        try:
            import joblib
            self.xgb       = joblib.load(f"{MODELS_DIR}/xgb.pkl")
            self.lgbm      = joblib.load(f"{MODELS_DIR}/lgbm.pkl")
            self.feat_cols = joblib.load(f"{MODELS_DIR}/features.pkl")
            print(f"ML models loaded ({len(self.feat_cols)} features)")
        except FileNotFoundError:
            print("No ML models yet — rule-based signals active")
            print("After 30+ days of data: run the ML training cell")


ai_engine = AISignalEngine()
print("AISignalEngine ready")

## Cell 9 — Fetch Live Data RIGHT NOW

In [ ]:
print("=" * 50)
print("  FETCHING LIVE DATA FROM NEPSEALPHA")
print("=" * 50)

# Step 1: Live market prices (your working endpoint)
live_data = scraper.fetch_live_market()
print(f"\\nLive market: {len(live_data)} stocks fetched")
if live_data:
    print("Sample (first 5):")
    for s in live_data[:5]:
        print(f"  {s['symbol']:8s}  LTP: {s['ltp']:8.2f}  "
              f"Change: {s['pct_change']:+.2f}%  Vol: {s['volume']:,}")

# Step 2: Indices
print("\\nFetching indices...")
n = scraper.fetch_indices()
print(f"Indices: {n} snapshots")

# Step 3: Top movers
print("\\nFetching top movers...")
n = scraper.fetch_top_movers()
print(f"Top movers: {n} entries")

# Step 4: Generate rule-based signals
print("\\nGenerating signals...")
signals = ai_engine.rule_based_signals()

print(f"\\n{'='*50}")
print(f"  SIGNALS SUMMARY")
print(f"{'='*50}")
buy_sigs  = [s for s in signals if s["signal"]=="BUY"]
sell_sigs = [s for s in signals if s["signal"]=="SELL"]

print(f"  BUY:  {len(buy_sigs)}")
print(f"  SELL: {len(sell_sigs)}")
print(f"  HOLD: {len(signals)-len(buy_sigs)-len(sell_sigs)}")

if buy_sigs:
    print(f"\\n  TOP BUY SIGNALS:")
    for s in buy_sigs[:10]:
        print(f"    {s['symbol']:8s} | Conf: {s['confidence_pct']:5.1f}% | "
              f"LTP: Rs.{s['ltp']:8.2f} | "
              f"SL: Rs.{s['stop_loss']:8.2f} | "
              f"TP: Rs.{s['target_price']:8.2f}")
        if s['key_drivers']:
            print(f"             Reasons: {' | '.join(s['key_drivers'][:3])}")

print(f"\\n  Signals saved to: {EXPORTS_DIR}/")

## Cell 10 — First-Time Historical Pull (run ONCE ~30 min)

In [ ]:
# !! ONLY RUN THIS ONCE — pulls 3 years of historical data
# Takes 20-40 minutes depending on number of stocks

print("=" * 50)
print("  FIRST-TIME HISTORICAL DATA PULL")
print("  This will take 20-40 minutes.")
print("  Do NOT disconnect Colab during this.")
print("=" * 50)

# Corporate actions first (dividends, bonus, AGM)
print("\\n[1/3] Corporate actions...")
n = scraper.fetch_corporate_actions()
print(f"  Done: {n} records")

# Historical OHLCV
print("\\n[2/3] Historical OHLCV (3 years)...")
print("  This is the long step...")
total_candles = scraper.fetch_all_history()
print(f"  Done: {total_candles:,} candles saved")

# Compute indicators on historical data
print("\\n[3/3] Computing technical indicators...")
n = indicator_engine.compute_all()
print(f"  Done: {n} stocks processed")

print("\\n" + "=" * 50)
print("  SETUP COMPLETE!")
print(f"  Database: {DB_PATH}")
print("  Now run Cell 11 for the live scheduler")
print("=" * 50)

## Cell 11 — Continuous Scheduler

In [ ]:
import schedule, time

print("=" * 50)
print("  NEPSE AI SCHEDULER STARTED")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("  Stop: Kernel > Interrupt")
print("=" * 50)


def job_live_market():
    if not is_market_open(): return
    logger.info("[SCHED] Live market fetch")
    scraper.fetch_live_market()

def job_indices():
    if not is_market_open(): return
    logger.info("[SCHED] Indices fetch")
    scraper.fetch_indices()

def job_top_movers():
    if not is_market_open(): return
    logger.info("[SCHED] Top movers")
    scraper.fetch_top_movers()

def job_eod():
    if not is_trading_day(): return
    logger.info("[SCHED] EOD OHLCV save")
    scraper.save_eod_ohlcv()

def job_floorsheet():
    if not is_trading_day(): return
    logger.info("[SCHED] Floorsheet + broker analysis")
    floor_scraper.fetch_floorsheet()
    floor_scraper.compute_broker_stats()

def job_indicators():
    if not is_trading_day(): return
    logger.info("[SCHED] Computing indicators")
    indicator_engine.compute_all()

def job_signals():
    if not is_trading_day(): return
    logger.info("[SCHED] Generating AI signals")
    sigs = ai_engine.rule_based_signals()
    buy  = sum(1 for s in sigs if s["signal"]=="BUY")
    logger.info(f"  Generated: {len(sigs)} signals | {buy} BUY")

def job_backtest():
    logger.info("[SCHED] Updating backtest accuracy")
    ai_engine.update_backtest()

def job_corp_actions():
    logger.info("[SCHED] Corporate actions")
    scraper.fetch_corporate_actions()


# ── Schedule setup ────────────────────────────────────────────
schedule.clear()

# During market hours (Sun–Thu, 11 AM–3 PM NPT)
schedule.every(5).minutes.do(job_live_market)
schedule.every(5).minutes.do(job_indices)
schedule.every(15).minutes.do(job_top_movers)

# End of day (times in NPT)
schedule.every().day.at("15:10").do(job_eod)
schedule.every().day.at("15:30").do(job_floorsheet)
schedule.every().day.at("15:45").do(job_indicators)
schedule.every().day.at("16:00").do(job_signals)
schedule.every().day.at("16:30").do(job_backtest)

# Daily
schedule.every().day.at("06:00").do(job_corp_actions)

print("\\nSchedule set:")
print("  Every 5 min  : Live market + Indices (market hours)")
print("  Every 15 min : Top movers (market hours)")
print("  15:10 NPT    : Save EOD OHLCV")
print("  15:30 NPT    : Full floorsheet + broker analysis")
print("  15:45 NPT    : Technical indicators")
print("  16:00 NPT    : AI signals generated + exported to Drive")
print("  06:00 NPT    : Corporate actions refresh")
print("\\nRunning... (Ctrl+C or Kernel > Interrupt to stop)\\n")

# Run immediately
job_live_market()
job_top_movers()

# Keep-alive loop for Colab
iteration = 0
while True:
    schedule.run_pending()
    iteration += 1
    if iteration % 60 == 0:  # Print status every 5 minutes
        cnt = db.one("SELECT COUNT(*) as c FROM live_market")
        sig = db.one("SELECT COUNT(*) as c FROM ai_signals WHERE signal_date=?",
                     [today_str()])
        print(f"[{datetime.now().strftime('%H:%M:%S')}] "
              f"Market records: {cnt['c']:,} | "
              f"Today's signals: {sig['c'] if sig else 0} | "
              f"Market open: {is_market_open()}")
    time.sleep(5)

## Cell 12 — View Signals & Export to Drive

In [ ]:
import pandas as pd

# ── Today's BUY signals ───────────────────────────────────────
print("=" * 60)
print("  TODAY'S BUY SIGNALS")
print("=" * 60)

buys = db.all(\"\"\"
    SELECT symbol, confidence_pct, ltp, stop_loss, target_price,
           risk_reward, key_drivers
    FROM ai_signals
    WHERE signal_date=? AND signal='BUY'
    ORDER BY confidence_pct DESC
\"\"\", [today_str()])

if buys:
    for s in buys:
        drivers = json.loads(s["key_drivers"]) if s.get("key_drivers") else []
        print(f"\\n  {s['symbol']:8s} | Conf: {s['confidence_pct']:5.1f}% | "
              f"LTP: Rs.{s['ltp']:,.2f}")
        print(f"           SL:  Rs.{s['stop_loss']:,.2f}  |  "
              f"TP: Rs.{s['target_price']:,.2f}  |  R:R 1:{s['risk_reward']:.1f}")
        if drivers:
            print(f"           Why: {' | '.join(drivers[:3])}")
else:
    print("  No BUY signals today. Run Cell 9 to generate.")

# ── Database summary ──────────────────────────────────────────
print("\\n" + "=" * 60)
print("  DATABASE SUMMARY")
print("=" * 60)

tables = ["stocks","daily_ohlcv","floorsheet","broker_concentration",
          "technical_indicators","ai_signals","news"]
for t in tables:
    r = db.one(f"SELECT COUNT(*) as c FROM {t}")
    print(f"  {t:<30s} {r['c']:>10,} rows")

# ── Export today's full signal sheet to Excel ─────────────────
print("\\n" + "=" * 60)
print("  EXPORTING SIGNAL REPORT")
print("=" * 60)

all_sigs = db.all(\"\"\"
    SELECT s.symbol, st.sector, s.signal, s.confidence_pct,
           s.ltp, s.stop_loss, s.target_price, s.risk_reward,
           s.key_drivers, s.signal_date, s.target_date
    FROM ai_signals s
    LEFT JOIN stocks st ON st.symbol=s.symbol
    WHERE s.signal_date=?
    ORDER BY CASE s.signal WHEN 'BUY' THEN 0 WHEN 'SELL' THEN 1 ELSE 2 END,
             s.confidence_pct DESC
\"\"\", [today_str()])

if all_sigs:
    df = pd.DataFrame(all_sigs)
    path = f"{EXPORTS_DIR}/full_report_{today_str()}.csv"
    df.to_csv(path, index=False)
    print(f"  Full report saved: {path}")
    print(f"  Total signals: {len(df)}")
    print(df[["symbol","signal","confidence_pct","ltp",
              "stop_loss","target_price","risk_reward"]].head(20).to_string(index=False))
else:
    print("  No signals to export. Run Cell 9 first.")

# ── Backtest results ──────────────────────────────────────────
print("\\n" + "=" * 60)
print("  BACKTEST ACCURACY (past signals)")
print("=" * 60)
ai_engine.update_backtest()